In [1]:
# Cell 1: Install dependencies
!pip install transformers -q

In [2]:
# Cell 2: Imports & config

import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import Counter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Config
SEED = 42
MODEL_NAME = "ehsanaghaei/SecureBERT"
MAX_LENGTH = 256
BATCH_SIZE = 128         # Kaggle T4 has 16GB — 64 is safe
GRAD_ACCUM_STEPS = 8     # effective batch = 256
EPOCHS = 10
LR = 1e-5               # lower than before to prevent collapse
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
THRESHOLD = 0.45
TIER_WEIGHTS = {"gold": 1.0, "gold+transitive": 0.6, "transitive": 0.2}

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Paths — adjust to your Kaggle dataset path
DATA_DIR = "/kaggle/input/datasets/ogoud073/csi-project-dataset"  # change if different
PARQUET_PATH = f"{DATA_DIR}/cve_attack_dataset.parquet"
META_PATH = f"{DATA_DIR}/dataset_metadata.pkl"

print("✅ Config ready.")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
✅ Config ready.


In [3]:
# Cell 3: Load dataset + metadata

dataset_df = pd.read_parquet(PARQUET_PATH)
with open(META_PATH, "rb") as f:
    meta = pickle.load(f)

tech2idx = meta["tech2idx"]
idx2tech = meta["idx2tech"]
tech_names = meta["tech_names"]
parent_techniques = meta["parent_techniques"]
NUM_CLASSES = meta["NUM_CLASSES"]
pos_weights = np.array(meta["pos_weights"])

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)

print(f"Dataset: {len(dataset_df):,} rows")
print(f"Classes: {NUM_CLASSES}")
print(f"Columns: {list(dataset_df.columns)}")
print(f"Tier distribution:\n{dataset_df['tier'].value_counts()}")
print(f"\nWeight stats: min={pos_weights.min():.3f}, max={pos_weights.max():.3f}, "
      f"mean={pos_weights.mean():.3f}")
print("✅ Data loaded.")

Dataset: 77,609 rows
Classes: 137
Columns: ['cve_id', 'description', 'cwes', 'techniques', 'num_techniques', 'tier', 'noisy', 'cvss_score', 'year']
Tier distribution:
tier
transitive         77190
gold                 295
gold+transitive      124
Name: count, dtype: int64

Weight stats: min=0.627, max=10.000, mean=5.589
✅ Data loaded.


In [4]:
# Cell 4: Train/val/test split

train_df, temp_df = train_test_split(
    dataset_df, test_size=0.2, random_state=SEED, stratify=dataset_df["tier"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["tier"]
)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {name}: {df['tier'].value_counts().to_dict()}")

# Verify no leakage
assert len(set(train_df["cve_id"]) & set(val_df["cve_id"])) == 0
assert len(set(train_df["cve_id"]) & set(test_df["cve_id"])) == 0
assert len(set(val_df["cve_id"]) & set(test_df["cve_id"])) == 0
print("✅ No CVE leakage.")

Train: 62,087  Val: 7,761  Test: 7,761
  Train: {'transitive': 61752, 'gold': 236, 'gold+transitive': 99}
  Val: {'transitive': 7719, 'gold': 29, 'gold+transitive': 13}
  Test: {'transitive': 7719, 'gold': 30, 'gold+transitive': 12}
✅ No CVE leakage.


In [5]:
# Cell 5: Tokenizer + Dataset class

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CVEDataset(Dataset):
    def __init__(self, df, tokenizer, tech2idx, num_classes, max_length=MAX_LENGTH):
        self.descriptions = df["description"].tolist()
        self.techniques_list = df["techniques"].tolist()
        self.tiers = df["tier"].tolist()
        self.noisy = df["noisy"].tolist()
        self.tokenizer = tokenizer
        self.tech2idx = tech2idx
        self.num_classes = num_classes
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.descriptions[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = torch.zeros(self.num_classes, dtype=torch.float32)
        for t in self.techniques_list[idx]:
            if t in self.tech2idx:
                labels[self.tech2idx[t]] = 1.0

        tier = self.tiers[idx]
        weight = TIER_WEIGHTS.get(tier, 0.2)
        # Don't penalize noisy for now — let model learn first
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
            "sample_weight": torch.tensor(weight, dtype=torch.float32),
        }

train_dataset = CVEDataset(train_df, tokenizer, tech2idx, NUM_CLASSES)
val_dataset = CVEDataset(val_df, tokenizer, tech2idx, NUM_CLASSES)
test_dataset = CVEDataset(test_df, tokenizer, tech2idx, NUM_CLASSES)

sample = train_dataset[0]
print(f"input_ids: {sample['input_ids'].shape}, labels sum: {sample['labels'].sum().item():.0f}")
print("✅ Datasets created.")

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

input_ids: torch.Size([256]), labels sum: 10
✅ Datasets created.


In [6]:
# Cell 6: Model

class SecureBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

model = SecureBERTClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print("✅ Model ready.")

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 657, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/ehsanaghaei/SecureBERT/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.p

Using 2 GPUs!
Parameters: 125,341,577
✅ Model ready.


In [7]:
# Cell 7: DataLoaders + Loss + Optimizer + Scheduler

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
                         num_workers=2, pin_memory=True)

class TieredWeightedBCE(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")

    def forward(self, logits, labels, sample_weights):
        loss = self.bce(logits, labels)       # (B, C)
        loss = loss.mean(dim=1)               # (B,)
        return (loss * sample_weights).mean()

criterion = TieredWeightedBCE(pos_weight_tensor.to(DEVICE))

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

scaler = GradScaler()

print(f"Train batches: {len(train_loader)}, Total steps: {total_steps}, Warmup: {warmup_steps}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print("✅ Training setup complete.")

Train batches: 485, Total steps: 606, Warmup: 60
Effective batch: 1024
✅ Training setup complete.


In [8]:
# Cell 8: Training loop

def evaluate(model, loader, criterion, device, threshold=THRESHOLD):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            sample_weights = batch["sample_weight"].to(device)

            with autocast():
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels, sample_weights)

            total_loss += loss.item() * len(labels)
            preds = (torch.sigmoid(logits) > threshold).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    avg_loss = total_loss / len(loader.dataset)
    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, micro_f1, macro_f1


best_val_micro_f1 = 0
best_epoch = 0
history = {"train_loss": [], "val_loss": [], "val_micro_f1": [], "val_macro_f1": []}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val µF1':>7} | {'Val MF1':>7} | {'Best':>4}")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        sample_weights = batch["sample_weight"].to(DEVICE)

        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels, sample_weights)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM_STEPS:.4f}"})

    avg_train_loss = epoch_loss / len(train_loader)
    val_loss, val_micro_f1, val_macro_f1 = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_micro_f1"].append(val_micro_f1)
    history["val_macro_f1"].append(val_macro_f1)

    is_best = val_micro_f1 > best_val_micro_f1
    if is_best:
        best_val_micro_f1 = val_micro_f1
        best_epoch = epoch + 1
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.module.state_dict() if hasattr(model, "module") else model.state_dict(),
            "val_micro_f1": val_micro_f1,
            "val_macro_f1": val_macro_f1,
            "tech2idx": tech2idx,
            "idx2tech": idx2tech,
        }, "best_securebert_classifier.pt")

    print(f"{epoch+1:>5} | {avg_train_loss:>10.4f} | {val_loss:>8.4f} | {val_micro_f1:>7.4f} | {val_macro_f1:>7.4f} | {'  ✓' if is_best else ''}")

print(f"\nBest epoch: {best_epoch} (Val µF1={best_val_micro_f1:.4f})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train"); ax1.plot(history["val_loss"], label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss"); ax1.legend()
ax2.plot(history["val_micro_f1"], label="Micro F1"); ax2.plot(history["val_macro_f1"], label="Macro F1")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("F1"); ax2.set_title("Validation F1"); ax2.legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()
print("✅ Training complete.")

Epoch | Train Loss | Val Loss | Val µF1 | Val MF1 | Best
------------------------------------------------------------


Epoch 1/10:   0%|          | 0/485 [00:00<?, ?it/s]

    1 |     0.1444 |   0.1261 |  0.1067 |  0.0420 |   ✓


Epoch 2/10:   0%|          | 0/485 [00:00<?, ?it/s]

    2 |     0.1107 |   0.0881 |  0.1306 |  0.0065 |   ✓


Epoch 3/10:   0%|          | 0/485 [00:00<?, ?it/s]

    3 |     0.0809 |   0.0692 |  0.0000 |  0.0000 | 


Epoch 4/10:   0%|          | 0/485 [00:00<?, ?it/s]

    4 |     0.0663 |   0.0608 |  0.0000 |  0.0000 | 


Epoch 5/10:   0%|          | 0/485 [00:00<?, ?it/s]

In [ ]:
# Cell 9: Test evaluation

# Replace the load line with:
checkpoint = torch.load("best_securebert_classifier.pt", map_location=DEVICE)
base_model = model.module if hasattr(model, "module") else model
base_model.load_state_dict(checkpoint["model_state_dict"])

model.eval()
all_preds, all_labels, all_logits = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        with autocast():
            logits = model(input_ids, attention_mask)
        all_logits.append(logits.cpu())
        all_preds.append((torch.sigmoid(logits) > THRESHOLD).cpu().numpy())
        all_labels.append(batch["labels"].numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)
all_logits = torch.cat(all_logits, dim=0)

micro_p = precision_score(all_labels, all_preds, average="micro", zero_division=0)
micro_r = recall_score(all_labels, all_preds, average="micro", zero_division=0)
micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
hamming = np.mean(all_preds != all_labels)

print(f"\n=== Test Results ({len(all_preds):,} samples, {NUM_CLASSES} classes) ===")
print(f"Micro P:  {micro_p:.4f}")
print(f"Micro R:  {micro_r:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Hamming:  {hamming:.4f}")

per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
per_class_p = precision_score(all_labels, all_preds, average=None, zero_division=0)
per_class_r = recall_score(all_labels, all_preds, average=None, zero_division=0)
per_class_support = all_labels.sum(axis=0)

class_results = []
for i in range(NUM_CLASSES):
    class_results.append({
        "technique": idx2tech[i], "name": tech_names.get(idx2tech[i], "?"),
        "f1": per_class_f1[i], "precision": per_class_p[i],
        "recall": per_class_r[i], "support": int(per_class_support[i]),
    })
class_results.sort(key=lambda x: -x["f1"])

print(f"\n=== Top 15 Techniques ===")
print(f"{'TID':<10} {'Name':<35} {'F1':>6} {'P':>6} {'R':>6} {'Supp':>6}")
print("-" * 75)
for r in class_results[:15]:
    print(f"{r['technique']:<10} {r['name'][:34]:<35} {r['f1']:>6.3f} {r['precision']:>6.3f} {r['recall']:>6.3f} {r['support']:>6}")

print(f"\n=== Bottom 15 Techniques ===")
print(f"{'TID':<10} {'Name':<35} {'F1':>6} {'P':>6} {'R':>6} {'Supp':>6}")
print("-" * 75)
for r in class_results[-15:]:
    print(f"{r['technique']:<10} {r['name'][:34]:<35} {r['f1']:>6.3f} {r['precision']:>6.3f} {r['recall']:>6.3f} {r['support']:>6}")

zero_f1 = [r for r in class_results if r["f1"] == 0]
print(f"\nClasses with F1=0: {len(zero_f1)}")
print("✅ Test evaluation complete.")